In [1]:
# path setup: this notebook now lives in revision_scripts/, while the data paths (data/...) and the
# config / netmedgpt imports are relative to the repository root. Safe to re-run; delete if the
# notebook is moved back to the root.
import os, sys
if os.path.basename(os.getcwd()) == 'revision_scripts':
    os.chdir('..')
for p in (os.getcwd(), os.path.join(os.getcwd(), 'revision_scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)


# Walk coverage: all results

Shows every result of `walk_coverage_analysis.ipynb` in one place. It only reads the saved CSV files in `data/new_analysis/coverage/` (no GPU, no data loading), so it can be re-run at any time. Sections whose files do not exist yet are skipped.

1. Coverage of the walk corpus (nodes, edges, relations, degree, relation tokens)
2. Check 1: theory vs measurement
3. Check 3+: never-walked edges scored by the model (degree-matched, strict unseen, walk baselines)

Values are mean ± std over the seeds that were run (std is empty for a single seed).

In [2]:
import os
import re
import glob
import numpy as np
import pandas as pd
from IPython.display import display

OUT_DIR = "data/new_analysis/coverage"
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)

GROUP_ORDER = ['seen', 'not adjacent (all)', 'not adjacent, same walk', 'not adjacent, never in same walk', 'test']


def path(name):
    return os.path.join(OUT_DIR, name)


# per-seed CSVs of every {inference}_seed{seed} folder, with inference and seed columns
def per_run(name):
    parts = []
    for p in sorted(glob.glob(os.path.join(OUT_DIR, "*_seed*", name))):
        m = re.match(r"(.+)_seed(\d+)$", os.path.basename(os.path.dirname(p)))
        parts.append(pd.read_csv(p).assign(inference=m.group(1), seed=int(m.group(2))))
    return pd.concat(parts, ignore_index=True) if parts else None


# "mean ± std" table over seeds
def mean_std(df, keys, cols, decimals=3):
    g = df.groupby(keys, sort=False, observed=True)[cols]
    m, s = g.mean().round(decimals), g.std().round(decimals)
    return m.astype(str) + s.map(lambda v: "" if pd.isna(v) else f" ± {v}")


def missing(name):
    print(f"Not available yet: {name}")

## 1. Coverage of the walk corpus

Measured on the cached walks NetMedGPT is trained on, against the training graph (val/test edges are excluded from the walks by design).

In [3]:
if os.path.exists(path("summary_mean_std.csv")):
    s = pd.read_csv(path("summary_mean_std.csv"))
    s['value'] = s['mean'].round(4).astype(str) + s['std'].map(lambda v: "" if pd.isna(v) or v == 0 else f" ± {round(v, 4)}")
    display(s.pivot_table(index=['section', 'metric'], columns='inference', values='value', aggfunc='first', sort=False))
else:
    missing("summary_mean_std.csv")

inference                                                                random_link_split         zero_shot_split
section            metric                                                                                         
corpus             walks                                              3684739.0 ± 701.7802    3685520.4 ± 631.5792
                   walk_steps                                       14738956.0 ± 2807.1209  14742081.6 ± 2526.3168
nodes              kg_nodes                                                       129375.0                129375.0
                   training_graph_nodes                                 123218.8 ± 21.5801      123213.4 ± 18.8759
                   covered_nodes                                        123173.4 ± 22.3002       123165.6 ± 18.743
                   covered_pct_of_kg_nodes                                95.2065 ± 0.0172        95.2005 ± 0.0145
                   covered_pct_of_training_graph_nodes                    99.9632 ± 0.0019        99.9612 ± 0.0031
edges              kg_directed_pairs                                             8099284.0               8099284.0
                   heldout_val_edges_excluded_by_design                           961938.0    961942.0 ± 1466.9547
                   heldout_test_edges_excluded_by_design                          405024.0     404232.4 ± 736.5805
                   training_triples                                     6697848.2 ± 5.5857    6700350.8 ± 802.9456
                   training_directed_pairs                             6697424.0 ± 11.4673    6699897.6 ± 801.9827
                   training_undirected_pairs                            3366342.0 ± 4.8477    3366706.8 ± 775.4087
                   directed_pair_coverage_pct                             59.0473 ± 0.0182         58.859 ± 0.0181
                   undirected_pair_coverage_pct                           77.3509 ± 0.0274        77.0577 ± 0.0193
                   exact_triple_coverage_pct                              59.0425 ± 0.0182        58.8543 ± 0.0182
                   mean_traversals_per_directed_pair                       2.2007 ± 0.0004         2.2003 ± 0.0004
                   median_traversals_per_directed_pair                                 1.0                     1.0
relation_direction training_triples_with_reverse_same_token_pct              99.4709 ± 0.0        99.4967 ± 0.0113
degree             mean_neighbourhood_completeness_pct                    97.1364 ± 0.0044        97.1167 ± 0.0064
                   spearman_node_occurrences_vs_degree                     0.6053 ± 0.0008         0.6056 ± 0.0006
metapaths          two_hop_patterns_in_training_graph                                259.0                   272.0
                   two_hop_patterns_observed                                257.2 ± 0.4472          269.0 ± 0.7071
                   two_hop_pattern_coverage_pct                            99.305 ± 0.1727          98.8971 ± 0.26
                   two_hop_patterns_in_walks_not_in_training_graph                     0.0                     0.0
                   spearman_walk_count_vs_graph_paths                      0.8719 ± 0.0005         0.8677 ± 0.0005
relation_tokens    kg_pairs_with_multiple_relations                                  842.0                   842.0
                   training_pairs_with_multiple_relations                   423.6 ± 7.3348          452.4 ± 13.813
                   walk_steps_off_training_pairs                                       0.0                     0.0
                   walk_steps_token_not_training_relation                  348.0 ± 38.8716         258.2 ± 49.4338
                   walk_steps_token_not_training_relation_pct              0.0024 ± 0.0003         0.0018 ± 0.0003
                   walk_steps_token_is_heldout_relation                    348.0 ± 38.8716         258.2 ± 49.4338

### Coverage per relation (sorted by undirected coverage)

In [4]:
if os.path.exists(path("relation_coverage_all_seeds.csv")):
    r = pd.read_csv(path("relation_coverage_all_seeds.csv"))
    cols = ['training_triples', 'undirected_coverage_pct', 'directed_coverage_pct', 'exact_coverage_pct', 'mean_traversals',
            'share_of_training_triples_pct', 'share_of_walk_steps_pct', 'reverse_with_same_token_pct']
    t = r.groupby(['inference', 'relation'], sort=False)[cols].mean().round(2)
    display(t.sort_values(['inference', 'undirected_coverage_pct']))
else:
    missing("relation_coverage_all_seeds.csv")

training_triples  undirected_coverage_pct  directed_coverage_pct  exact_coverage_pct  mean_traversals  share_of_training_triples_pct  share_of_walk_steps_pct  reverse_with_same_token_pct
inference         relation                                                                                                                                                                                                              
random_link_split drug_drug                          2221622.0                    46.99                  27.37               27.37             0.34                          33.17                     5.09                        100.0
                  contraindication                     25498.0                    70.68                  70.68               70.37             2.34                           0.38                     0.40                          0.0
                  drug_effect                         107704.0                    83.20                  55.25               55.25             1.12                           1.61                     0.82                        100.0
                  indication                            7804.0                    86.84                  86.84               85.57             4.29                           0.12                     0.23                          0.0
                  drug_protein                         42397.2                    89.09                  66.24               66.24             3.90                           0.63                     1.12                        100.0
                  anatomy_protein_present            2524012.0                    90.40                  69.16               69.16             1.28                          37.68                    21.98                        100.0
                  off-label use                         2135.0                    91.91                  91.91               91.91             5.43                           0.03                     0.08                          0.0
                  protein_protein                     533786.0                    92.33                  72.88               72.88             1.43                           7.97                     5.17                        100.0
                  anatomy_protein_absent               33062.0                    92.87                  71.95               71.94             1.56                           0.49                     0.35                        100.0
                  cellcomp_protein                    138656.0                    95.86                  80.37               80.37             2.32                           2.07                     2.18                        100.0
                  molfunc_protein                     115594.0                    96.24                  81.60               81.60             3.13                           1.73                     2.45                        100.0
                  pathway_protein                      70900.0                    96.32                  81.80               81.80             2.42                           1.06                     1.16                        100.0
                  disease_protein                     133682.0                    96.79                  82.60               82.60             2.51                           2.00                     2.28                        100.0
                  phenotype_protein                     5536.0                    98.25                  87.08               87.08             2.65                           0.08                     0.10                        100.0
                  bioprocess_protein                  240738.0                    98.39                  87.14               87.14             3.74                           3.59                     6.10                        100.0
                  exposure_protein                      2014.0                    99.

### Coverage by node degree

In [5]:
if os.path.exists(path("degree_coverage_all_seeds.csv")):
    d = pd.read_csv(path("degree_coverage_all_seeds.csv"))
    cols = ['nodes', 'mean_degree', 'node_coverage_pct', 'mean_neighbourhood_completeness_pct',
            'median_neighbourhood_completeness_pct', 'mean_occurrences']
    display(mean_std(d, ['inference', 'degree_bin'], cols, 2))
else:
    missing("degree_coverage_all_seeds.csv")

nodes     mean_degree node_coverage_pct mean_neighbourhood_completeness_pct median_neighbourhood_completeness_pct mean_occurrences
inference         degree_bin                                                                                                                                              
random_link_split 1           32501.6 ± 37.39       1.0 ± 0.0      99.91 ± 0.01                        99.91 ± 0.01                           100.0 ± 0.0     65.84 ± 0.09
                  2-5         39056.8 ± 66.04      3.05 ± 0.0      99.97 ± 0.01                         99.9 ± 0.01                           100.0 ± 0.0     89.58 ± 0.07
                  6-20        20613.8 ± 78.74    10.57 ± 0.02      99.99 ± 0.01                        99.57 ± 0.01                           100.0 ± 0.0    120.81 ± 0.12
                  21-100      16422.0 ± 19.07    49.85 ± 0.07      99.99 ± 0.01                         95.4 ± 0.01                           97.9 ± 0.02    189.72 ± 0.27
                  101-1000     13677.2 ± 8.93   233.05 ± 0.18       100.0 ± 0.0                        83.96 ± 0.04                          90.26 ± 0.04    340.39 ± 0.19
                  >1000          947.4 ± 3.05  2488.02 ± 4.77       100.0 ± 0.0                        55.64 ± 0.06                          48.13 ± 0.09    2663.64 ± 5.5
zero_shot_split   1           32520.6 ± 36.46       1.0 ± 0.0      99.91 ± 0.01                        99.91 ± 0.01                           100.0 ± 0.0     65.91 ± 0.09
                  2-5         39074.6 ± 68.49      3.05 ± 0.0      99.96 ± 0.01                         99.9 ± 0.01                           100.0 ± 0.0     89.76 ± 0.08
                  6-20        20587.4 ± 79.96    10.57 ± 0.02      99.99 ± 0.01                        99.58 ± 0.01                           100.0 ± 0.0    121.26 ± 0.11
                  21-100      16401.8 ± 15.37    49.87 ± 0.07      99.99 ± 0.01                         95.4 ± 0.03                          97.92 ± 0.04     190.28 ± 0.3
                  101-1000    13681.4 ± 11.26   233.08 ± 0.14       100.0 ± 0.0                         83.8 ± 0.02                          90.23 ± 0.03    339.64 ± 0.22
                  >1000          947.6 ± 1.82  2487.88 ± 3.27       100.0 ± 0.0                        54.93 ± 0.08                          47.31 ± 0.06   2651.77 ± 4.32

### Coverage per node type

In [6]:
nt = per_run("node_type_coverage.csv")
if nt is not None:
    display(mean_std(nt, ['inference', 'node_type'],
                     ['kg_nodes', 'training_graph_nodes', 'covered_nodes', 'covered_pct_of_kg_nodes',
                      'covered_pct_of_training_graph_nodes'], 2))
else:
    missing("node_type_coverage.csv")

kg_nodes training_graph_nodes    covered_nodes covered_pct_of_kg_nodes covered_pct_of_training_graph_nodes
inference         node_type                                                                                                                          
random_link_split anatomy             14035.0 ± 0.0      12147.6 ± 16.98  12138.6 ± 16.43            86.49 ± 0.12                        99.93 ± 0.01
                  biological_process  28642.0 ± 0.0      27862.6 ± 24.36  27862.4 ± 24.25            97.28 ± 0.08                         100.0 ± 0.0
                  cellular_component   4176.0 ± 0.0       3866.0 ± 14.04   3865.8 ± 14.24            92.57 ± 0.34                        99.99 ± 0.01
                  disease             17080.0 ± 0.0      16488.0 ± 13.58  16487.0 ± 13.58            96.53 ± 0.08                         99.99 ± 0.0
                  drug                 7957.0 ± 0.0       7469.4 ± 17.42   7437.0 ± 18.41            93.46 ± 0.23                        99.57 ± 0.03
                  effect/phenotype    15311.0 ± 0.0      14657.0 ± 18.41  14657.0 ± 18.41            95.73 ± 0.12                         100.0 ± 0.0
                  exposure              818.0 ± 0.0         783.6 ± 2.97     783.6 ± 2.97            95.79 ± 0.36                         100.0 ± 0.0
                  gene/protein        27671.0 ± 0.0      27224.0 ± 13.69  27221.6 ± 13.94            98.38 ± 0.05                         99.99 ± 0.0
                  molecular_function  11169.0 ± 0.0      10220.4 ± 17.62  10220.4 ± 17.62            91.51 ± 0.16                         100.0 ± 0.0
                  pathway              2516.0 ± 0.0        2500.2 ± 3.56    2500.0 ± 3.94            99.36 ± 0.16                        99.99 ± 0.02
zero_shot_split   anatomy             14035.0 ± 0.0      12147.6 ± 16.98  12138.6 ± 16.43            86.49 ± 0.12                        99.93 ± 0.01
                  biological_process  28642.0 ± 0.0      27862.6 ± 24.36  27862.4 ± 24.25            97.28 ± 0.08                         100.0 ± 0.0
                  cellular_component   4176.0 ± 0.0       3866.0 ± 14.04   3865.8 ± 14.24            92.57 ± 0.34                        99.99 ± 0.01
                  disease             17080.0 ± 0.0      16483.0 ± 14.14  16481.6 ± 14.08             96.5 ± 0.08                         99.99 ± 0.0
                  drug                 7957.0 ± 0.0       7469.0 ± 22.37   7434.8 ± 20.54            93.44 ± 0.26                        99.54 ± 0.05
                  effect/phenotype    15311.0 ± 0.0      14657.0 ± 18.41  14657.0 ± 18.41            95.73 ± 0.12                         100.0 ± 0.0
                  exposure              818.0 ± 0.0         783.6 ± 2.97     783.6 ± 2.97            95.79 ± 0.36                         100.0 ± 0.0
                  gene/protein        27671.0 ± 0.0      27224.0 ± 13.69  27221.4 ± 13.59            98.38 ± 0.05                         99.99 ± 0.0
                  molecular_function  11169.0 ± 0.0      10220.4 ± 17.62  10220.4 ± 17.62            91.51 ± 0.16                         100.0 ± 0.0
                  pathway              2516.0 ± 0.0        2500.2 ± 3.56    2500.0 ± 3.94            99.36 ± 0.16                        99.99 ± 0.02

### Edge coverage by the degrees of both end nodes (hub-hub edges)

In [7]:
ed = per_run("edge_degree_coverage.csv")
if ed is not None:
    t = ed.groupby(['inference', 'min_endpoint_degree', 'max_endpoint_degree'], sort=False)[['undirected_pairs', 'coverage_pct']].mean().round(1)
    display(t)
else:
    missing("edge_degree_coverage.csv")

undirected_pairs  coverage_pct
inference         min_endpoint_degree max_endpoint_degree                                
random_link_split 1                   1                               244.0         100.0
                                      2-5                            8269.2         100.0
                                      6-20                           7358.8         100.0
                                      21-100                         6988.8          99.8
                                      101-1000                       7107.6         100.0
                                      >1000                          2289.2          99.6
                  2-5                 2-5                           14841.6         100.0
                                      6-20                          25224.8          99.9
                                      21-100                        25040.2          99.8
                                      101-1000                      22108.8          99.9
                                      >1000                          8617.6         100.0
                  6-20                6-20                          14758.8          99.8
                                      21-100                        49925.2          99.5
                                      101-1000                      73106.8          99.4
                                      >1000                         32817.8          98.6
                  21-100              21-100                        62842.2          98.2
                                      101-1000                     241974.4          94.8
                                      >1000                        369095.4          91.2
                  101-1000            101-1000                     648514.0          68.2
                                      >1000                       1546115.4          74.1
                  >1000               >1000                        199101.4          45.6
zero_shot_split   1                   1                               244.4         100.0
                                      2-5                            8284.0         100.0
                                      6-20                           7360.4         100.0
                                      21-100                         6991.2          99.8
                                      101-1000                       7107.4          99.9
                                      >1000                          2288.8          99.6
                  2-5                 2-5                           14865.4         100.0
                                      6-20                          25243.4          99.9
                                      21-100                        25035.8          99.8
                                      101-1000                      22095.8          99.9
                                      >1000                          8607.0         100.0
                  6-20                6-20                          14742.2          99.9
                                      21-100                        49939.6          99.5
                                      101-1000                      72963.0          99.4
                                      >1000                         32579.0          98.6
                  21-100              21-100                        62759.8          98.3
                                      101-1000                     241639.6          94.8
                                      >1000                        368790.6          91.2
                  101-1000            101-1000                     649020.6          67.8
                                      >1000                       1547050.4          73.8
                  >1000               >1000                        199098.4          44.7

### Relation-token check (per seed)

Walk steps whose relation token is not a training relation of that node pair (node pairs with several relations keep one token); `token_is_heldout_relation` = the token is a val/test relation of the pair.

In [8]:
tc = per_run("relation_token_check.csv")
if tc is not None:
    n_seeds = tc.groupby('inference')['seed'].nunique()
    t = (tc.groupby(['inference', 'token_relation', 'training_relations', 'token_is_heldout_relation'])
           .agg(node_pairs=('walk_steps', 'size'), walk_steps=('walk_steps', 'sum')).reset_index())
    t[['node_pairs', 'walk_steps']] = t[['node_pairs', 'walk_steps']].div(t['inference'].map(n_seeds), axis=0).round(1)
    display(t)
else:
    missing("relation_token_check.csv")

,inference,token_relation,training_relations,token_is_heldout_relation,node_pairs,walk_steps
0,random_link_split,anatomy_protein_present,anatomy_protein_absent,True,0.8,2.4
1,random_link_split,disease_phenotype_positive,disease_phenotype_negative,True,48.2,263.0
2,random_link_split,indication,contraindication,True,7.0,14.4
3,random_link_split,off-label use,contraindication,True,4.2,10.4
4,random_link_split,off-label use,contraindication;indication,True,0.4,3.2
5,random_link_split,off-label use,indication,True,12.2,54.6
6,zero_shot_split,anatomy_protein_present,anatomy_protein_absent,True,0.8,2.2
7,zero_shot_split,disease_phenotype_positive,disease_phenotype_negative,True,48.4,256.0


## 2. Check 1: theory vs measurement

Coverage predicted from the training graph alone (expected walk steps per edge from the transition matrix; an edge is missed with probability exp(−λ)) vs the coverage measured in the walks.

In [9]:
for name, keys in [('overall', ['metric']), ('degree', ['degree_bin']), ('relation', ['relation'])]:
    p = path(f"check1_{name}_all_seeds.csv")
    if not os.path.exists(p):
        missing(p)
        continue
    df = pd.read_csv(p)
    print(name)
    t = df.drop(columns='seed').groupby(['inference'] + keys, sort=False).mean().round(3)
    if {'predicted_undirected_coverage_pct', 'measured_undirected_coverage_pct'} <= set(t.columns):
        t['difference'] = (t['measured_undirected_coverage_pct'] - t['predicted_undirected_coverage_pct']).round(2)
    elif {'predicted', 'measured'} <= set(t.columns):
        t['difference'] = (t['measured'] - t['predicted']).round(3)
    display(t)

overall


predicted     measured  difference
inference         metric                                                                   
random_link_split kept_walks                           3690367.979  3684739.000   -5628.979
                  dropped_walks_pct                          4.918          NaN         NaN
                  mean_traversals_per_directed_pair          2.204        2.201      -0.003
                  directed_pair_coverage_pct                59.114       59.047      -0.067
                  undirected_pair_coverage_pct              77.438       77.351      -0.087
                  mean_neighbourhood_completeness_pct       97.225       97.136      -0.089
                  spearman_node_occurrences_vs_degree        0.613        0.605      -0.008
zero_shot_split   kept_walks                           3691158.118  3685520.400   -5637.718
                  dropped_walks_pct                          4.898          NaN         NaN
                  mean_traversals_per_directed_pair          2.204        2.200      -0.004
                  directed_pair_coverage_pct                58.922       58.859      -0.063
                  undirected_pair_coverage_pct              77.146       77.058      -0.088
                  mean_neighbourhood_completeness_pct       97.212       97.117      -0.095
                  spearman_node_occurrences_vs_degree        0.614        0.606      -0.008

degree


nodes  predicted_neighbourhood_completeness_pct  measured_neighbourhood_completeness_pct  predicted_mean_occurrences  measured_mean_occurrences
inference         degree_bin                                                                                                                                                   
random_link_split 1           32501.6                                    99.931                                   99.911                      65.908                     65.844
                  2-5         39056.8                                    99.926                                   99.896                      89.693                     89.581
                  6-20        20613.8                                    99.663                                   99.566                     121.062                    120.810
                  21-100      16422.0                                    95.807                                   95.403                     190.948                    189.716
                  101-1000    13677.2                                    83.993                                   83.956                     339.498                    340.387
                  >1000         947.4                                    55.634                                   55.638                    2672.505                   2663.640
zero_shot_split   1           32520.6                                    99.935                                   99.908                      65.952                     65.909
                  2-5         39074.6                                    99.929                                   99.900                      89.838                     89.758
                  6-20        20587.4                                    99.688                                   99.584                     121.507                    121.263
                  21-100      16401.8                                    95.830                                   95.402                     191.601                    190.275
                  101-1000    13681.4                                    83.838                                   83.799                     338.815                    339.640
                  >1000         947.6                                    54.922                                   54.932                    2660.476                   2651.769

relation


training_triples  predicted_undirected_coverage_pct  measured_undirected_coverage_pct  difference
inference         relation                                                                                                                     
random_link_split anatomy_protein_present            2524012.0                             90.544                            90.395       -0.15
                  protein_protein                     533786.0                             92.437                            92.326       -0.11
                  bioprocess_protein                  240738.0                             98.544                            98.394       -0.15
                  anatomy_protein_absent               33062.0                             93.101                            92.874       -0.23
                  molfunc_protein                     115594.0                             96.368                            96.241       -0.13
                  cellcomp_protein                    138656.0                             96.068                            95.858       -0.21
                  disease_protein                     133682.0                             96.853                            96.789       -0.06
                  pathway_protein                      70900.0                             96.458                            96.324       -0.13
                  drug_protein                         42397.2                             89.356                            89.094       -0.26
                  phenotype_protein                     5536.0                             98.377                            98.251       -0.13
                  exposure_protein                      2014.0                             99.884                            99.762       -0.12
                  drug_drug                          2221622.0                             47.000                            46.985       -0.02
                  drug_effect                         107704.0                             83.290                            83.197       -0.09
                  contraindication                     25498.0                             70.526                            70.678        0.15
                  indication                            7804.0                             86.551                            86.838        0.29
                  off-label use                         2135.0                             91.622                            91.906        0.28
                  disease_phenotype_positive          249902.0                             99.914                            99.910       -0.00
                  phenotype_phenotype                  31148.0                             99.971                            99.964       -0.01
                  disease_phenotype_negative            1982.0                             99.959                            99.899       -0.06
                  disease_disease                      53522.0                             99.821                            99.842        0.02
                  exposure_disease                      3830.0                             99.930                            99.927       -0.00
                  bioprocess_bioprocess                87924.0                             99.990                            99.989       -0.00
                  exposure_bioprocess                   2702.0                             99.989                            99.985       -0.00
                  molfunc_molfunc                      22566.0                             99.963                            99.965        0.00
                  exposure_molfunc                        76.0                            100.000                           100.000        0.00
                  cellcomp_cellcomp                     8056.0                             99.953                            99.921       -0.03
              

## 3. Check 3+: never-walked edges scored by the model

Per-edge filtered rank among all candidate tails (AUROC per edge, Hits@100).
- a) `*_degree_matched`: every group re-weighted to the node degrees of the never-walked edges.
- b) `not adjacent, never in same walk`: the two nodes never appear together in any walk.
- c) walk baselines on the same edges (walk_coverage_baselines.ipynb, seed 1).

In [10]:
if os.path.exists(path("check3x_summary_all_seeds.csv")):
    s3 = pd.read_csv(path("check3x_summary_all_seeds.csv"))
    s3['group'] = pd.Categorical(s3['group'], categories=GROUP_ORDER, ordered=True)
    s3 = s3.sort_values(['inference', 'relation', 'method', 'group'])

    print("a/b) NetMedGPT")
    nm = s3[s3['method'] == 'NetMedGPT']
    display(mean_std(nm, ['inference', 'relation', 'group'],
                     ['auroc', 'auroc_degree_matched', 'hits100', 'hits100_degree_matched']))
    display(nm.groupby(['inference', 'relation', 'group'], sort=False, observed=True)[['edges', 'matched_weight_pct']].mean().round(1))
else:
    missing("check3x_summary_all_seeds.csv")

a/b) NetMedGPT


auroc auroc_degree_matched        hits100 hits100_degree_matched
inference         relation         group                                                                                                     
random_link_split contraindication seen                                0.994 ± 0.0          0.992 ± 0.0  0.871 ± 0.003          0.833 ± 0.006
                                   not adjacent (all)                0.975 ± 0.002        0.975 ± 0.002  0.582 ± 0.005          0.582 ± 0.005
                                   not adjacent, same walk           0.984 ± 0.001        0.983 ± 0.002  0.637 ± 0.022          0.651 ± 0.043
                                   not adjacent, never in same walk  0.974 ± 0.002        0.975 ± 0.002  0.578 ± 0.004           0.58 ± 0.003
                                   test                              0.983 ± 0.001         0.98 ± 0.002  0.681 ± 0.008          0.643 ± 0.009
                  drug_effect      seen                                0.997 ± 0.0          0.998 ± 0.0  0.877 ± 0.006          0.914 ± 0.004
                                   not adjacent (all)                  0.998 ± 0.0          0.998 ± 0.0  0.906 ± 0.005          0.906 ± 0.005
                                   not adjacent, same walk             0.999 ± 0.0          0.998 ± 0.0  0.954 ± 0.005          0.938 ± 0.008
                                   not adjacent, never in same walk    0.997 ± 0.0          0.998 ± 0.0  0.894 ± 0.007          0.899 ± 0.006
                                   test                                0.996 ± 0.0          0.997 ± 0.0  0.828 ± 0.009           0.88 ± 0.008
                  drug_protein     seen                                0.998 ± 0.0          0.998 ± 0.0  0.927 ± 0.003          0.906 ± 0.005
                                   not adjacent (all)                0.987 ± 0.001        0.987 ± 0.001  0.844 ± 0.006          0.844 ± 0.006
                                   not adjacent, same walk             0.999 ± 0.0          0.999 ± 0.0  0.972 ± 0.003          0.964 ± 0.009
                                   not adjacent, never in same walk  0.984 ± 0.002        0.985 ± 0.002  0.819 ± 0.006          0.823 ± 0.006
                                   test                              0.987 ± 0.002        0.993 ± 0.002  0.778 ± 0.013           0.828 ± 0.02
                  indication       seen                                0.998 ± 0.0          0.997 ± 0.0  0.964 ± 0.003          0.955 ± 0.003
                                   not adjacent (all)                0.934 ± 0.005        0.934 ± 0.005  0.682 ± 0.009          0.682 ± 0.009
                                   not adjacent, same walk           0.986 ± 0.004        0.985 ± 0.006  0.769 ± 0.043          0.785 ± 0.072
                                   not adjacent, never in same walk   0.93 ± 0.005        0.933 ± 0.005  0.675 ± 0.007          0.679 ± 0.008
                                   test                              0.974 ± 0.006        0.967 ± 0.009  0.819 ± 0.018           0.793 ± 0.02
                  off-label use    seen                                0.998 ± 0.0        0.997 ± 0.001  0.975 ± 0.006          0.958 ± 0.012
                                   not adjacent (all)                 0.98 ± 0.006         0.98 ± 0.006  0.754 ± 0.039          0.754 ± 0.039
                                   not adjacent, same walk           0.993 ± 0.004        0.991 ± 0.008  0.841 ± 0.143          0.793 ± 0.223
                                   not adjacent, never in same walk  0.978 ± 0.006        0.979 ± 0.006  0.744 ± 0.049          0.747 ± 0.049
                                   test                              0.993 ± 0.002        0.992 ± 0.004  0.892 ± 0.041          0.883 ± 0.037
zero_shot_split   contraindication seen                              0.993 ± 0.001        0.991 ± 0.001  0.845 ± 0.021          0.796 ± 0.028
                                   not adjacent (all)                0.976 ± 0.001        0.976 ± 0.0

edges  matched_weight_pct
inference         relation         group                                                       
random_link_split contraindication seen                              5000.0                99.9
                                   not adjacent (all)                5000.0               100.0
                                   not adjacent, same walk            373.4                97.2
                                   not adjacent, never in same walk  4626.6               100.0
                                   test                              1534.0                99.9
                  drug_effect      seen                              5000.0               100.0
                                   not adjacent (all)                5000.0               100.0
                                   not adjacent, same walk           1036.0                99.8
                                   not adjacent, never in same walk  3964.0               100.0
                                   test                              3239.0               100.0
                  drug_protein     seen                              5000.0               100.0
                                   not adjacent (all)                2312.0               100.0
                                   not adjacent, same walk            381.6                98.4
                                   not adjacent, never in same walk  1930.4               100.0
                                   test                              1283.0                99.9
                  indication       seen                              5000.0                99.9
                                   not adjacent (all)                1027.2               100.0
                                   not adjacent, same walk             70.4                87.8
                                   not adjacent, never in same walk   956.8               100.0
                                   test                               469.0                98.6
                  off-label use    seen                              1962.2                99.8
                                   not adjacent (all)                 172.8               100.0
                                   not adjacent, same walk             11.2                71.5
                                   not adjacent, never in same walk   161.6                99.9
                                   test                               128.0                97.4
zero_shot_split   contraindication seen                              5000.0               100.0
                                   not adjacent (all)                5000.0               100.0
                                   not adjacent, same walk            394.6                96.0
                                   not adjacent, never in same walk  4605.4               100.0
                                   test                              1133.2                99.2
                  indication       seen                              5000.0                99.9
                                   not adjacent (all)                1146.2               100.0
                                   not adjacent, same walk             75.4                87.2
                                   not adjacent, never in same walk  1070.8               100.0
                                   test                               465.0                98.4

### b) Share of never-walked edges whose two nodes never appear in the same walk

In [11]:
if os.path.exists(path("check3x_edges_all_seeds.csv")):
    e3 = pd.read_csv(path("check3x_edges_all_seeds.csv"))
    e3 = e3[(e3['method'] == 'NetMedGPT') & e3['group'].str.startswith('not adjacent')]
    share = (e3.assign(never=e3['group'] == 'not adjacent, never in same walk')
               .groupby(['inference', 'seed', 'relation'])['never'].mean().mul(100))
    display(mean_std(share.reset_index(), ['inference', 'relation'], ['never'], 1).rename(columns={'never': 'never_in_same_walk_pct'}))
else:
    missing("check3x_edges_all_seeds.csv")

never_in_same_walk_pct
inference         relation                               
random_link_split contraindication             92.5 ± 0.3
                  drug_effect                  79.3 ± 0.4
                  drug_protein                 83.5 ± 1.0
                  indication                   93.1 ± 0.7
                  off-label use                93.4 ± 2.6
zero_shot_split   contraindication             92.1 ± 0.6
                  indication                   93.4 ± 1.0

### c) NetMedGPT vs walk baselines on the same edges (degree-matched)

In [12]:
# walk baselines (node2vec, DREAMwalk same corpus) on the same edges, from walk_coverage_baselines.ipynb
cb = per_run("check3x_baselines_summary.csv")
if cb is None:
    missing("check3x_baselines_summary.csv")
else:
    cb['group'] = pd.Categorical(cb['group'], categories=GROUP_ORDER, ordered=True)
    for v in ['auroc_degree_matched', 'hits100_degree_matched']:
        print(v)
        display(cb.pivot_table(index=['inference', 'seed', 'relation', 'group'], columns='method', values=v,
                               aggfunc='mean', observed=True).round(3))

auroc_degree_matched


method                                                                    NetMedGPT  dreamwalk_samecorpus  node2vec_classifier  node2vec_rel_classifier
inference         seed relation         group                                                                                                          
random_link_split 1    contraindication seen                                  0.992                 0.974                0.927                    0.959
                                        not adjacent (all)                    0.976                 0.867                0.867                    0.936
                                        not adjacent, same walk               0.980                 0.907                0.873                    0.949
                                        not adjacent, never in same walk      0.976                 0.865                0.866                    0.935
                                        test                                  0.982                 0.866                0.873                    0.943
                       drug_effect      seen                                  0.998                 0.998                0.996                    0.997
                                        not adjacent (all)                    0.998                 0.997                0.997                    0.998
                                        not adjacent, same walk               0.998                 0.998                0.998                    0.998
                                        not adjacent, never in same walk      0.997                 0.997                0.997                    0.997
                                        test                                  0.997                 0.995                0.995                    0.997
                       drug_protein     seen                                  0.998                 0.991                0.990                    0.983
                                        not adjacent (all)                    0.985                 0.943                0.972                    0.974
                                        not adjacent, same walk               0.999                 0.991                0.998                    0.997
                                        not adjacent, never in same walk      0.983                 0.936                0.968                    0.971
                                        test                                  0.990                 0.935                0.973                    0.968
                       indication       seen                                  0.997                 0.982                0.945                    0.949
                                        not adjacent (all)                    0.933                 0.805                0.856                    0.878
                                        not adjacent, same walk               0.987                 0.865                0.906                    0.933
                                        not adjacent, never in same walk      0.932                 0.803                0.855                    0.876
                                        test                                  0.955                 0.824                0.890                    0.902
                       off-label use    seen                                  0.998                 0.996                0.953                    0.963
                                        not adjacent (all)                    0.983                 0.827                0.840                    0.897
                                        not adjacent, same walk               0.985                 0.529                0.826                    0.904
                                        not adjacent, never in same walk      0.983                 0.835                0.841                    0.897
                                        test    

hits100_degree_matched


method                                                                    NetMedGPT  dreamwalk_samecorpus  node2vec_classifier  node2vec_rel_classifier
inference         seed relation         group                                                                                                          
random_link_split 1    contraindication seen                                  0.834                 0.560                0.434                    0.484
                                        not adjacent (all)                    0.584                 0.288                0.267                    0.369
                                        not adjacent, same walk               0.658                 0.295                0.315                    0.451
                                        not adjacent, never in same walk      0.581                 0.287                0.263                    0.366
                                        test                                  0.644                 0.307                0.308                    0.400
                       drug_effect      seen                                  0.909                 0.902                0.858                    0.866
                                        not adjacent (all)                    0.905                 0.877                0.865                    0.898
                                        not adjacent, same walk               0.936                 0.916                0.919                    0.943
                                        not adjacent, never in same walk      0.898                 0.867                0.854                    0.890
                                        test                                  0.869                 0.838                0.806                    0.857
                       drug_protein     seen                                  0.910                 0.704                0.768                    0.683
                                        not adjacent (all)                    0.840                 0.678                0.717                    0.700
                                        not adjacent, same walk               0.958                 0.866                0.936                    0.922
                                        not adjacent, never in same walk      0.821                 0.646                0.682                    0.664
                                        test                                  0.798                 0.594                0.673                    0.611
                       indication       seen                                  0.955                 0.638                0.598                    0.546
                                        not adjacent (all)                    0.678                 0.242                0.288                    0.299
                                        not adjacent, same walk               0.777                 0.238                0.289                    0.334
                                        not adjacent, never in same walk      0.676                 0.242                0.288                    0.297
                                        test                                  0.796                 0.314                0.424                    0.392
                       off-label use    seen                                  0.961                 0.931                0.655                    0.636
                                        not adjacent (all)                    0.755                 0.328                0.276                    0.302
                                        not adjacent, same walk               0.581                 0.304                0.304                    0.202
                                        not adjacent, never in same walk      0.759                 0.325                0.271                    0.302
                                        test    